In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import os
import subprocess
import time
import pandas
import pickle
import math
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.metrics import accuracy_score, silhouette_score, adjusted_rand_score, silhouette_samples
from sklearn.cluster import AgglomerativeClustering, SpectralClustering, KMeans
from sklearn.model_selection import KFold, LeaveOneOut, train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.kernel_ridge import KernelRidge
from sklearn import linear_model
from sklearn.manifold import TSNE
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import (ModelDesc, EvalEnvironment, Term, EvalFactor, LookupFactor, dmatrices, INTERCEPT)
from statsmodels.distributions.empirical_distribution import ECDF
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.colorbar as colorbar
import sys
import re

In [2]:
sns.set_style('ticks')
import matplotlib as mpl
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['legend.labelspacing'] = 0.2
mpl.rcParams['axes.labelpad'] = 2
mpl.rcParams['xtick.major.size'] = 2
mpl.rcParams['xtick.major.width'] = 0.5
mpl.rcParams['xtick.major.pad'] = 1
mpl.rcParams['ytick.major.size'] = 2
mpl.rcParams['ytick.major.width'] = 0.5
mpl.rcParams['ytick.major.pad'] = 1
mpl.rcParams['lines.scale_dashes'] = False
mpl.rcParams['lines.dashed_pattern'] = (2, 1)
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['text.color'] = 'k'

## General functions

In [3]:
def standardize_plot_graphics(ax):
    [i.set_linewidth(0.5) for i in ax.spines.values()]
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    return ax

def fit_regression(x, y):
    lm = sm.OLS(y, sm.add_constant(x)).fit()
    x_range = sm.add_constant(np.array([x.min(), x.max()]))
    x_range_pred = lm.predict(x_range)
    return lm.pvalues[1], lm.params[1], x_range[:,1], x_range_pred, lm.rsquared

def CDFplot(x, ax, **kwargs):
    x = np.array(x)
    ix=np.argsort(x)
    ax.plot(x[ix], ECDF(x)(x)[ix], **kwargs)
    return ax

def fit_regression_and_plot(x, y, ax, plot_label='', color='k', linecolor='r', markersize=3,
                            show_pval=True):
    #linetype is a string like 'bo'
    pvalue, slope, temp, temppred, R2 = fit_regression(x, y)   
    if show_pval:
        plot_label = '%s p=%.2e\nr=%.3f'% (plot_label, pvalue, np.sign(slope)*np.sqrt(R2))
    else:
        plot_label = '%s r=%.3f'% (plot_label, np.sign(slope)*np.sqrt(R2))
    ax.scatter(x, y, color=color, label=plot_label, s=markersize)
    ax.plot(temp, temppred, color=linecolor)
    return ax, slope, pvalue, R2

## Clustering set up

In [ ]:
basedir = '' # folder containing code and data
populationdata = np.load(os.path.join(basedir, ''))

In [ ]:
print(populationdata.shape)

Now, we will visualize the full dataset to get a sense of what they look like. As mentioned previously, the first 100 features correspond to CS+ responses and the last 100 features correspond to CS- responses. So I will plot both trial types separately.

In [6]:
framerate = 1  
window_size = 240  

sortwindow = [60, 63] # coresponding to 1st airpuff 
sortresponse = np.argsort(np.mean(populationdata[:, sortwindow[0]:sortwindow[1]], axis=1))[::-1]
cmax = 3.0

In [ ]:
# Plotting the heatmap and time series
fig, axs = plt.subplots(2, 1, figsize=(6, 6), sharex=True)
cbar_ax = fig.add_axes([0.91, 0.3, 0.02, 0.4])
cbar_ax.tick_params(width=0.5)

ax = axs[0]
sns.heatmap(populationdata[sortresponse, :],
            ax=ax,
            cmap=plt.get_cmap('coolwarm'),
            vmin=-cmax,
            vmax=cmax,
            cbar=True,
            cbar_ax=cbar_ax,
            cbar_kws={'label': 'Z score'})
ax.grid(False)
ax.tick_params(width=0.5)

xticks = [0, 60, 120, 180, 240]
ax.set_xticks(xticks)
ax.set_xticklabels([str(t) for t in xticks])
ax.set_yticks([])
for t_event in [60, 120, 180]:
    ax.axvline(t_event, linestyle='--', color='k', linewidth=0.5)
ax.set_xlabel('Time (s)')

ax = axs[1]
mean_time_series = np.mean(populationdata[sortresponse, :], axis=0)
sem_time_series = np.std(populationdata[sortresponse, :], axis=0) / np.sqrt(populationdata[sortresponse, :].shape[0])

ax.plot(np.arange(window_size), mean_time_series, color='blue', label='Mean response')
ax.fill_between(np.arange(window_size),
                mean_time_series - sem_time_series,
                mean_time_series + sem_time_series,
                color='blue', alpha=0.3, label='SEM')

for t_event in [60, 120, 180]:
    ax.axvline(t_event, linestyle='--', color='k', linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_xticks(xticks)
ax.set_xticklabels([str(t) for t in xticks])
axs[0].set_ylabel('Neurons')
axs[1].set_ylabel('Mean norm. fluor.')
ax.tick_params(width=0.5)

fig.tight_layout()
fig.subplots_adjust(right=0.88)


Reduce the dimensionality of the response feature space for each neuron using principal component analysis.

In [ ]:
load_savedpca_or_dopca = 'dopca'
# Select 'dopca' for doing PCA on the data. Select 'savedpca' for loading my previous results

if load_savedpca_or_dopca == 'dopca':
    pca = PCA(n_components=None, whiten=True)
    pca.fit(populationdata) 
    with open(os.path.join(basedir, 'pcaresults.pickle'), 'wb') as f:
        pickle.dump(pca, f)
elif load_savedpca_or_dopca == 'savedpca':
    with open(os.path.join(basedir, 'pcaresults.pickle'), 'rb') as f:
        pca = pickle.load(f)
    
transformed_data = pca.transform(populationdata)

pca_vectors = pca.components_
print('Number of PCs = %d' % (pca_vectors.shape[0]))

x = 100*pca.explained_variance_ratio_
xprime = x - (x[0] + (x[-1]-x[0])/(x.size-1)*np.arange(x.size))
num_retained_pcs = np.argmin(xprime)
print('Number of PCs to keep = %d' % (num_retained_pcs))


fig, ax = plt.subplots(figsize=(2,2))
ax.plot(np.arange(pca.explained_variance_ratio_.shape[0]).astype(int)+1, x, 'k')
ax.set_ylabel('Percentage of\nvariance explained')
ax.set_xlabel('PC number')
ax.axvline(num_retained_pcs, linestyle='--', color='k', linewidth=0.5)
ax.set_title('Scree plot')
ax.set_xlim([0,50])
[i.set_linewidth(0.5) for i in ax.spines.values()]
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

fig.subplots_adjust(left=0.3)
fig.subplots_adjust(right=0.98)
fig.subplots_adjust(bottom=0.25)
fig.subplots_adjust(top=0.9)

trial_types = ['All']
colors_for_key = {'All': (0, 0.5, 1)}



numcols = 5.0
fig, axs = plt.subplots(int(np.ceil(num_retained_pcs/numcols)), int(numcols), sharey='all',
                        figsize=(2*numcols, 2*int(np.ceil(num_retained_pcs/numcols))))
for pc in range(num_retained_pcs):
    ax = axs.flat[pc]

    ax.plot(pca_vectors[pc, :window_size], color=colors_for_key['All'],
            label='PC %d'%(pc+1))
    for t_event in [60, 120, 180]:
        ax.axvline(t_event, linestyle='--', color='k', linewidth=0.5) 
    ax.annotate('PC %d'%(pc+1), xy=(0.45, 0.06), xycoords='axes fraction',
            textcoords='axes fraction', multialignment='center', size='large')
    
    xticks = [0, window_size//2, window_size]
    ax.set_xticks(xticks)
    ax.set_xticklabels([str(t) for t in xticks])
    ax.set_yticks([])
    [i.set_linewidth(0.5) for i in ax.spines.values()]
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

fig.text(0.5, 0.05, 'Time (s)', horizontalalignment='center', rotation='horizontal')
fig.text(0.02, 0.6, 'PCA weights', verticalalignment='center', rotation='vertical')
fig.tight_layout()
for ax in axs.flat[num_retained_pcs:]:
    ax.set_visible(False)

fig.subplots_adjust(wspace=0.08, hspace=0.08)
fig.subplots_adjust(bottom=0.13)

In [17]:
transformed_data = pca.transform(populationdata)
np.save(os.path.join(basedir, 'Ctrl_transformed_data_temp.npy'), transformed_data)

## Perform clustering

In [ ]:
max_n_clusters = 20 

possible_n_clusters = np.arange(2, max_n_clusters+1) 

possible_n_nearest_neighbors = np.array([50, 100, 200, 250]) # This should be selected for each dataset appropriately. 
    
silhouette_scores = np.nan*np.ones((possible_n_clusters.size,
                                    possible_n_nearest_neighbors.size))

for n_clustersidx, n_clusters in enumerate(possible_n_clusters):
    for nnidx, nn in enumerate(possible_n_nearest_neighbors):
        model = SpectralClustering(n_clusters=n_clusters, affinity='nearest_neighbors', n_neighbors=nn)
        model.fit(transformed_data[:,:num_retained_pcs])
        silhouette_scores[n_clustersidx, nnidx] = silhouette_score(transformed_data[:,:num_retained_pcs],
                                                                   model.labels_,
                                                                   metric='cosine')
        print ('Done with numclusters = %d, num nearest neighbors = %d: score = %.3f'%(n_clusters,
                                                                                      nn,
                                                                                      silhouette_scores[n_clustersidx,                                                                           
                                                                                                        nnidx]))

print ('Done with model fitting')

temp = {}
temp['possible_n_clusters'] = possible_n_clusters
temp['possible_n_nearest_neighbors'] = possible_n_nearest_neighbors
temp['silhouette_scores'] = silhouette_scores
temp['shape'] = 'cluster_nn'
with open(os.path.join(basedir, 'silhouette_scores.pickle'), 'wb') as f:
    pickle.dump(temp, f)

Next, we will reorder the cluster labels such that there is a fixed order for naming the clusters. This is important because if you run the previous code multiple times, it will return different orders for the cluster labels.

In [ ]:
with open(os.path.join(basedir, 'silhouette_scores.pickle'), 'rb') as f:
    silhouette_scores = pickle.load(f)
    
transformed_data = np.load(os.path.join(basedir, 'Ctrl_transformed_data.npy'))

# Identify optimal parameters from the above parameter space
temp = np.where(silhouette_scores['silhouette_scores'] == np.nanmax(silhouette_scores['silhouette_scores']))
n_clusters = silhouette_scores['possible_n_clusters'][temp[0][0]]
n_nearest_neighbors = silhouette_scores['possible_n_nearest_neighbors'][temp[1][0]]

print(n_clusters, n_nearest_neighbors)

# Redo clustering with these optimal parameters
model = SpectralClustering(n_clusters=n_clusters,
                           affinity='nearest_neighbors',
                           n_neighbors=n_nearest_neighbors)

model.fit(transformed_data[:, :num_retained_pcs])

temp_score = silhouette_score(transformed_data[:, :num_retained_pcs], model.labels_, metric='cosine')
print('Number of clusters = %d, average silhouette = %.3f' % (len(set(model.labels_)), temp_score))

# Save this optimal clustering model.
with open(os.path.join(basedir, 'clusteringmodel.pickle'), 'wb') as f:
     pickle.dump(model, f)


# Create a new variable containing all unique cluster labels
uniquelabels = list(set(model.labels_))

np.save(os.path.join(basedir, 'CombinedRemapped_clusterlabels.npy'), model.labels_)


Make a silhouette plot to visualize clustering quality. The average silhouette score above is the mean of all sample silhouettes. There are two other clustering methods commented out above (KMeans and agglomerative). One can see that if you apply those methods, the average silhouette score is lower, demonstrating that those methods are worse for these data. This also then shows that the high dimensional geometry for these data is unlikely to be a simple Gaussian structure, as these other methods are specialized for picking up Gaussian clusters. Spectral clustering is effective as it can uncover peculiarly shaped clusters

In [184]:
def make_silhouette_plot(X, cluster_labels):
    
    n_clusters = len(set(cluster_labels))
    
    fig, ax = plt.subplots(1, 1)
    fig.set_size_inches(4, 4)

    # The 1st subplot is the silhouette plot
    # The silhouette coefficient can range from -1, 1 but in this example all
    # lie within [-0.1, 1]
    ax.set_xlim([-0.4, 1])
    # The (n_clusters+1)*10 is for inserting blank space between silhouette
    # plots of individual clusters, to demarcate them clearly.
    ax.set_ylim([0, len(X) + (n_clusters + 1) * 10])
    silhouette_avg = silhouette_score(X, cluster_labels, metric='cosine')

    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X, cluster_labels, metric='cosine')

    y_lower = 10
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to
        # cluster i, and sort them
        ith_cluster_silhouette_values = \
            sample_silhouette_values[cluster_labels == i]

        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = colors_for_cluster[i]
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                          0, ith_cluster_silhouette_values,
                          facecolor=color, edgecolor=color, alpha=0.9)

        # Label the silhouette plots with their cluster numbers at the middle
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i+1))

        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    ax.set_title("The silhouette plot for the various clusters.")
    ax.set_xlabel("The silhouette coefficient values")
    ax.set_ylabel("Cluster label")

    # The vertical line for average silhouette score of all the values
    ax.axvline(x=silhouette_avg, color="red", linestyle="--")

    ax.set_yticks([])  # Clear the yaxis labels / ticks
    ax.set_xticks([-0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1])

In [ ]:
make_silhouette_plot(transformed_data[:,:num_retained_pcs], model.labels_)

Plot the identified clusters. This plot is a good comparison to the original plot of the data, which did not cluster the data

In [ ]:
cmax = 3.0
sortwindow = [60, 63]

fig, axs = plt.subplots(1, len(uniquelabels), figsize=(2*len(uniquelabels), 2))

cbar_ax = fig.add_axes([0.94, 0.3, 0.01, 0.4])
cbar_ax.tick_params(width=0.5) 

numroisincluster = np.nan * np.ones(len(uniquelabels))

for c, cluster in enumerate(uniquelabels):
    temp = populationdata[model.labels_ == cluster, :]
    numroisincluster[c] = temp.shape[0]
    
    sortresponse = np.argsort(np.mean(temp[:, sortwindow[0]:sortwindow[1]], axis=1))[::-1]
    

    sns.heatmap(temp[sortresponse],
                ax=axs[c],
                cmap='coolwarm',
                vmin=-cmax,
                vmax=cmax,
                cbar=(c == 0), 
                cbar_ax=cbar_ax if (c == 0) else None,
                cbar_kws={'label': 'Normalized Fluor.'})
    
    axs[c].grid(False)
    axs[c].tick_params(width=0.5)
    axs[c].set_yticks([])
    for t_event in [60, 120, 180]:
        axs[c].axvline(t_event, linestyle='--', color='k', linewidth=0.5)
    axs[c].set_title(f'Cluster {cluster+1}\n(n={int(numroisincluster[c])})')
    xticks = [0, 60, 120, 180, 239]
    axs[c].set_xticks(xticks)
    axs[c].set_xticklabels([str(t) for t in xticks])


fig.tight_layout()
fig.subplots_adjust(wspace=0.1, hspace=0.1, left=0.03, right=0.93, bottom=0.2, top=0.83)
